In [ ]:
%matplotlib inline
import numpy as np
import scipy.signal as scipy_sig
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import soundfile as sf
import pandas as pd
import time
import csv
from pathlib import Path
from abc import ABC, abstractmethod

BASE_DIR = Path('.')
RAW_DIR = BASE_DIR / 'data' / 'raw'
PROCESSED_DIR = BASE_DIR / 'data' / 'processed'
IR_DIR = BASE_DIR / 'data' / 'ir'
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
SR = 44100
plt.rcParams['figure.dpi'] = 100
print('Imports OK')

## Section 1: Effect Classes (self-contained)

All effect classes are reproduced here for notebook independence.

In [ ]:
class AudioEffect(ABC):
    def __init__(self, name='effect'):
        self.name = name; self.bypass = False
    @abstractmethod
    def process(self, signal, sample_rate): pass
    def get_params(self):
        return {k:v for k,v in self.__dict__.items()
                if not k.startswith('_') and k!='name' and not callable(v)
                and not isinstance(v, np.ndarray)}

class SchroederReverb(AudioEffect):
    def __init__(self, room_size=0.5, damping=0.5, wet_mix=0.5, pre_delay_ms=20.0):
        super().__init__('schroeder_reverb')
        self.room_size=float(room_size); self.damping=float(damping)
        self.wet_mix=float(wet_mix); self.pre_delay_ms=float(pre_delay_ms)
    def _comb(self, x, d, fb, dp):
        d=max(1,int(d)); buf=np.zeros(d); y=np.zeros(len(x)); fs=0.0; pos=0
        d2=1-dp
        for i in range(len(x)):
            o=buf[pos]; fs=o*d2+fs*dp; buf[pos]=x[i]+fs*fb; y[i]=o; pos=(pos+1)%d
        return y
    def _ap(self, x, d, g=0.5):
        d=max(1,int(d)); buf=np.zeros(d); y=np.zeros(len(x)); pos=0
        for i in range(len(x)):
            b=buf[pos]; buf[pos]=x[i]+b*g; y[i]=b-x[i]*g; pos=(pos+1)%d
        return y
    def process(self, signal, sample_rate):
        dry=signal.astype(np.float32)
        pd=int(self.pre_delay_ms*0.001*sample_rate)
        x=np.concatenate([np.zeros(pd),dry])[:len(dry)]
        sc=0.5+self.room_size*0.5
        cds=[int(d*sc*sample_rate/1000) for d in [29.7,37.1,41.1,43.7]]
        fb=0.7+self.room_size*0.18
        co=sum(self._comb(x,cd,fb,self.damping*0.4) for cd in cds)*0.25
        for ad in [int(5.0*sample_rate/1000),int(1.7*sample_rate/1000)]:
            co=self._ap(co,ad)
        return np.clip((1-self.wet_mix)*dry+self.wet_mix*co,-1,1).astype(np.float32)

class DigitalDelay(AudioEffect):
    def __init__(self, delay_ms=300.0, feedback=0.4, wet_mix=0.4):
        super().__init__('digital_delay')
        self.delay_ms=float(delay_ms); self.feedback=float(feedback); self.wet_mix=float(wet_mix)
    def process(self, signal, sample_rate):
        dry=signal.astype(np.float32); ds=max(1,int(self.delay_ms*sample_rate/1000))
        dl=np.zeros(ds); wet=np.zeros_like(dry); pos=0
        for i in range(len(dry)):
            d=dl[pos]; wet[i]=d; dl[pos]=dry[i]+d*self.feedback; pos=(pos+1)%ds
        return np.clip((1-self.wet_mix)*dry+self.wet_mix*wet,-1,1).astype(np.float32)

class Distortion(AudioEffect):
    def __init__(self, drive=5.0, mode='soft', pre_gain=1.0, post_gain=0.5, wet_mix=1.0):
        super().__init__('distortion')
        self.drive=float(drive); self.mode=mode; self.pre_gain=float(pre_gain)
        self.post_gain=float(post_gain); self.wet_mix=float(wet_mix)
    def process(self, signal, sample_rate):
        dry=signal.astype(np.float32); x=dry*self.pre_gain*self.drive
        wet=np.tanh(x) if self.mode=='soft' else np.clip(x,-1,1)
        return np.clip((1-self.wet_mix)*dry+self.wet_mix*wet*self.post_gain,-1,1).astype(np.float32)

class Compressor(AudioEffect):
    def __init__(self, threshold_db=-20.0, ratio=4.0, attack_ms=10.0,
                 release_ms=100.0, knee_db=6.0, makeup_gain_db=6.0):
        super().__init__('compressor')
        self.threshold_db=float(threshold_db); self.ratio=float(ratio)
        self.attack_ms=float(attack_ms); self.release_ms=float(release_ms)
        self.knee_db=float(knee_db); self.makeup_gain_db=float(makeup_gain_db)
    def process(self, signal, sample_rate):
        x=signal.astype(np.float64)
        ac=np.exp(-1/(self.attack_ms*0.001*sample_rate))
        rc=np.exp(-1/(self.release_ms*0.001*sample_rate))
        T=self.threshold_db; R=self.ratio; W=self.knee_db
        mk=10**(self.makeup_gain_db/20)
        ldb=20*np.log10(np.abs(x)+1e-10)
        gdb=np.zeros_like(ldb)
        for i in range(len(ldb)):
            lv=ldb[i]
            if 2*(lv-T)<-W: gdb[i]=0
            elif 2*abs(lv-T)<=W: gdb[i]=(1/R-1)*(lv-T+W/2)**2/(2*W)
            else: gdb[i]=(1/R-1)*(lv-T)
        sg=np.zeros_like(gdb); prev=0.0
        for i in range(len(gdb)):
            sg[i]=ac*prev+(1-ac)*gdb[i] if gdb[i]<prev else rc*prev+(1-rc)*gdb[i]
            prev=sg[i]
        return np.clip(x*10**(sg/20)*mk,-1,1).astype(np.float32)

class ParametricEQ(AudioEffect):
    def __init__(self, low_shelf_freq=80.0, low_shelf_gain_db=0.0,
                 peak1_freq=250.0, peak1_gain_db=0.0, peak1_q=1.0,
                 peak2_freq=1000.0, peak2_gain_db=0.0, peak2_q=1.0,
                 peak3_freq=4000.0, peak3_gain_db=0.0, peak3_q=1.0,
                 high_shelf_freq=8000.0, high_shelf_gain_db=0.0):
        super().__init__('parametric_eq')
        self.low_shelf_freq=float(low_shelf_freq); self.low_shelf_gain_db=float(low_shelf_gain_db)
        self.peak1_freq=float(peak1_freq); self.peak1_gain_db=float(peak1_gain_db); self.peak1_q=float(peak1_q)
        self.peak2_freq=float(peak2_freq); self.peak2_gain_db=float(peak2_gain_db); self.peak2_q=float(peak2_q)
        self.peak3_freq=float(peak3_freq); self.peak3_gain_db=float(peak3_gain_db); self.peak3_q=float(peak3_q)
        self.high_shelf_freq=float(high_shelf_freq); self.high_shelf_gain_db=float(high_shelf_gain_db)
    def _ls(self,f,g,sr):
        A=10**(g/40); w0=2*np.pi*f/sr; alpha=np.sin(w0)/2*np.sqrt((A+1/A)*(1/0.707-1)+2)
        b0=A*((A+1)-(A-1)*np.cos(w0)+2*np.sqrt(A)*alpha); b1=2*A*((A-1)-(A+1)*np.cos(w0))
        b2=A*((A+1)-(A-1)*np.cos(w0)-2*np.sqrt(A)*alpha)
        a0=(A+1)+(A-1)*np.cos(w0)+2*np.sqrt(A)*alpha; a1=-2*((A-1)+(A+1)*np.cos(w0))
        a2=(A+1)+(A-1)*np.cos(w0)-2*np.sqrt(A)*alpha
        return [b0/a0,b1/a0,b2/a0],[1,a1/a0,a2/a0]
    def _hs(self,f,g,sr):
        A=10**(g/40); w0=2*np.pi*f/sr; alpha=np.sin(w0)/2*np.sqrt((A+1/A)*(1/0.707-1)+2)
        b0=A*((A+1)+(A-1)*np.cos(w0)+2*np.sqrt(A)*alpha); b1=-2*A*((A-1)+(A+1)*np.cos(w0))
        b2=A*((A+1)+(A-1)*np.cos(w0)-2*np.sqrt(A)*alpha)
        a0=(A+1)-(A-1)*np.cos(w0)+2*np.sqrt(A)*alpha; a1=2*((A-1)-(A+1)*np.cos(w0))
        a2=(A+1)-(A-1)*np.cos(w0)-2*np.sqrt(A)*alpha
        return [b0/a0,b1/a0,b2/a0],[1,a1/a0,a2/a0]
    def _pk(self,f,g,Q,sr):
        A=10**(g/40); w0=2*np.pi*f/sr; alpha=np.sin(w0)/(2*Q)
        b0=1+alpha*A; b1=-2*np.cos(w0); b2=1-alpha*A
        a0=1+alpha/A; a1=-2*np.cos(w0); a2=1-alpha/A
        return [b0/a0,b1/a0,b2/a0],[1,a1/a0,a2/a0]
    def process(self, signal, sample_rate):
        x=signal.astype(np.float32)
        for b,a in [self._ls(self.low_shelf_freq,self.low_shelf_gain_db,sample_rate),
                    self._pk(self.peak1_freq,self.peak1_gain_db,self.peak1_q,sample_rate),
                    self._pk(self.peak2_freq,self.peak2_gain_db,self.peak2_q,sample_rate),
                    self._pk(self.peak3_freq,self.peak3_gain_db,self.peak3_q,sample_rate),
                    self._hs(self.high_shelf_freq,self.high_shelf_gain_db,sample_rate)]:
            x=scipy_sig.lfilter(b,a,x).astype(np.float32)
        return np.clip(x,-1,1).astype(np.float32)

class BitCrusher(AudioEffect):
    def __init__(self, bit_depth=8, downsample=4, wet_mix=1.0):
        super().__init__('bit_crusher')
        self.bit_depth=max(1,min(16,int(bit_depth))); self.downsample=max(1,int(downsample)); self.wet_mix=float(wet_mix)
    def process(self, signal, sample_rate):
        dry=signal.astype(np.float32)
        held=dry[(np.arange(len(dry))//self.downsample)*self.downsample] if self.downsample>1 else dry.copy()
        lvls=2**self.bit_depth; q=np.round(held*(lvls/2))/(lvls/2)
        return np.clip((1-self.wet_mix)*dry+self.wet_mix*q,-1,1).astype(np.float32)

class Tremolo(AudioEffect):
    def __init__(self, rate_hz=5.0, depth=0.8, shape='sine', wet_mix=1.0):
        super().__init__('tremolo')
        self.rate_hz=float(rate_hz); self.depth=float(depth); self.shape=shape; self.wet_mix=float(wet_mix)
    def process(self, signal, sample_rate):
        dry=signal.astype(np.float32); t=np.arange(len(dry))/sample_rate
        lfo=np.sin(2*np.pi*self.rate_hz*t)
        env=1.0-self.depth*(0.5+0.5*lfo)
        return ((1-self.wet_mix)*dry + self.wet_mix*(dry*env.astype(np.float32))).astype(np.float32)

class Limiter(AudioEffect):
    def __init__(self, threshold_db=-1.0, lookahead_ms=5.0, release_ms=50.0):
        super().__init__('limiter')
        self.threshold_db=float(threshold_db); self.lookahead_ms=float(lookahead_ms); self.release_ms=float(release_ms)
    def process(self, signal, sample_rate):
        x=signal.astype(np.float64)
        threshold=10**(self.threshold_db/20)
        la=max(1,int(self.lookahead_ms*0.001*sample_rate))
        rc=np.exp(-1/(self.release_ms*0.001*sample_rate))
        delayed=np.concatenate([np.zeros(la),x])
        gain=np.ones(len(x)); pk=0.0
        for i in range(len(x)):
            we=min(i+la,len(x))
            p=np.max(np.abs(x[i:we]))
            pk=max(pk,threshold/(p+1e-10)) if p>threshold else min(1.0,pk+(1-pk)*(1-rc))
            gain[i]=pk
        gs=np.zeros_like(gain); prev=1.0
        for i in range(len(gain)):
            gs[i]=gain[i] if gain[i]<prev else rc*prev+(1-rc)*gain[i]
            prev=gs[i]
        out=delayed[la:la+len(x)]*gs
        return np.clip(out,-1,1).astype(np.float32)

print('All effects defined.')

# Instantiate all effects for benchmarking
ALL_EFFECTS = [
    SchroederReverb(room_size=0.5),
    DigitalDelay(delay_ms=200),
    Distortion(drive=5.0),
    Compressor(),
    ParametricEQ(low_shelf_gain_db=3, high_shelf_gain_db=-2),
    BitCrusher(bit_depth=8, downsample=4),
    Tremolo(rate_hz=4.0),
    Limiter(threshold_db=-3.0),
]
print(f'Effects ready: {[e.__class__.__name__ for e in ALL_EFFECTS]}')

## Section 2: Audio Quality Metrics

All metrics are implemented from scratch using numpy/scipy.

- **SNR**: ratio of signal power to noise power. Higher = cleaner.
- **THD**: ratio of harmonic distortion power to fundamental power.
- **Spectral centroid shift**: how the 'brightness' of the signal changes.
- **Crest factor**: ratio of peak to RMS — higher = more dynamic.
- **Dynamic range**: difference between max and min RMS over short windows.

In [ ]:
def compute_snr(original, processed):
    """SNR in dB: 10*log10(signal_power / noise_power)."""
    noise = processed.astype(np.float64) - original.astype(np.float64)
    signal_power = np.mean(original.astype(np.float64)**2)
    noise_power = np.mean(noise**2)
    if noise_power < 1e-20:
        return np.inf
    return 10 * np.log10(signal_power / noise_power)

def compute_thd(signal, sr, fundamental_freq=440.0, num_harmonics=8):
    """
    Total Harmonic Distortion in %.
    THD = sqrt(sum of harmonic powers) / fundamental power * 100.
    """
    N = len(signal)
    freqs = np.fft.rfftfreq(N, 1/sr)
    spectrum = np.abs(np.fft.rfft(signal.astype(np.float64)))
    bin_width = sr / N

    def get_peak_near(freq, window_hz=5):
        idx = int(freq / bin_width)
        hw = max(1, int(window_hz / bin_width))
        lo = max(0, idx - hw); hi = min(len(spectrum), idx + hw)
        return np.max(spectrum[lo:hi])

    fundamental_mag = get_peak_near(fundamental_freq)
    if fundamental_mag < 1e-10:
        return 0.0
    harmonic_sum_sq = sum(get_peak_near(fundamental_freq * (k+2))**2
                          for k in range(num_harmonics-1)
                          if fundamental_freq * (k+2) < sr/2)
    thd = np.sqrt(harmonic_sum_sq) / fundamental_mag * 100
    return float(thd)

def spectral_centroid(signal, sr):
    """Spectral centroid in Hz: weighted mean frequency."""
    N = len(signal)
    freqs = np.fft.rfftfreq(N, 1/sr)
    mag = np.abs(np.fft.rfft(signal.astype(np.float64)))
    total = np.sum(mag)
    if total < 1e-10:
        return 0.0
    return float(np.sum(freqs * mag) / total)

def crest_factor(signal):
    """Crest factor = peak / RMS. Higher = more peaked (more dynamic)."""
    rms = np.sqrt(np.mean(signal.astype(np.float64)**2))
    peak = np.max(np.abs(signal))
    if rms < 1e-10:
        return 0.0
    return float(peak / rms)

def dynamic_range_db(signal, sr, window_ms=100):
    """
    Dynamic range in dB: difference between loudest and quietest 100ms windows.
    """
    win = max(1, int(window_ms * 0.001 * sr))
    x = signal.astype(np.float64)
    rms_windows = []
    for start in range(0, len(x) - win, win // 2):
        chunk = x[start:start+win]
        rms_val = np.sqrt(np.mean(chunk**2))
        if rms_val > 1e-10:
            rms_windows.append(rms_val)
    if len(rms_windows) < 2:
        return 0.0
    return float(20 * np.log10(max(rms_windows) / min(rms_windows)))

def pesq_approximation(original, processed, sr):
    """
    PESQ approximation using log-spectral distance (wideband LLR).
    Not true ITU PESQ, but a widely-used spectral distance measure.
    Lower = more similar to original.
    """
    n_fft = 512; hop = 128
    def stft_mag(x):
        frames = []
        window = np.hanning(n_fft)
        for i in range(0, len(x) - n_fft, hop):
            frame = x[i:i+n_fft] * window
            frames.append(np.abs(np.fft.rfft(frame)))
        return np.array(frames) if frames else np.zeros((1, n_fft//2+1))
    orig_mag = stft_mag(original.astype(np.float64)) + 1e-10
    proc_mag = stft_mag(processed.astype(np.float64)) + 1e-10
    min_frames = min(len(orig_mag), len(proc_mag))
    if min_frames == 0:
        return 0.0
    log_diff = np.log(orig_mag[:min_frames]) - np.log(proc_mag[:min_frames])
    lsd = float(np.mean(np.sqrt(np.mean(log_diff**2, axis=1))))
    return lsd

def compute_all_metrics(dry, wet, sr, fundamental_freq=440.0):
    """Compute all quality metrics and return as dict."""
    n = min(len(dry), len(wet))
    dry_t = dry[:n]; wet_t = wet[:n]
    return {
        'snr_db': round(float(compute_snr(dry_t, wet_t)), 2),
        'thd_pct': round(compute_thd(wet_t, sr, fundamental_freq), 3),
        'pesq_approx': round(pesq_approximation(dry_t, wet_t, sr), 4),
        'centroid_dry_hz': round(spectral_centroid(dry_t, sr), 1),
        'centroid_wet_hz': round(spectral_centroid(wet_t, sr), 1),
        'centroid_shift_hz': round(spectral_centroid(wet_t, sr) - spectral_centroid(dry_t, sr), 1),
        'crest_dry': round(crest_factor(dry_t), 3),
        'crest_wet': round(crest_factor(wet_t), 3),
        'dynrange_dry_db': round(dynamic_range_db(dry_t, sr), 2),
        'dynrange_wet_db': round(dynamic_range_db(wet_t, sr), 2),
    }

# Quick test
t = np.linspace(0, 2.0, int(SR*2), endpoint=False)
test_dry = (0.7 * np.sin(2*np.pi*440*t)).astype(np.float32)
test_wet = Distortion(drive=5).process(test_dry, SR)
metrics = compute_all_metrics(test_dry, test_wet, SR)
print('Metrics test:')
for k, v in metrics.items():
    print(f'  {k:<25}: {v}')

## Section 3: Benchmark Suite

Each effect is timed over 100 iterations on 5 seconds of audio at 44100 Hz. The Real-Time Factor (RTF) = processing_time / audio_duration. RTF < 1.0 means the effect runs faster than real time.

In [ ]:
# Generate 5-second test signal
bench_duration = 5.0
n_bench = int(SR * bench_duration)
t_bench = np.linspace(0, bench_duration, n_bench, endpoint=False)
bench_signal = (0.5 * np.sin(2*np.pi*440*t_bench) + 0.1*np.random.randn(n_bench)).astype(np.float32)

BENCH_ITERS = 10  # Reduced for speed; increase to 100 for production
bench_results = []

for effect in ALL_EFFECTS:
    name = effect.__class__.__name__
    times = []
    for _ in range(BENCH_ITERS):
        t0 = time.perf_counter()
        _ = effect.process(bench_signal, SR)
        t1 = time.perf_counter()
        times.append(t1 - t0)
    mean_ms = np.mean(times) * 1000
    std_ms = np.std(times) * 1000
    rtf = np.mean(times) / bench_duration
    bench_results.append({
        'Effect': name,
        'Mean Time (ms)': round(mean_ms, 2),
        'Std Dev (ms)': round(std_ms, 2),
        'RTF': round(rtf, 4),
        'Realtime?': 'YES' if rtf < 1.0 else 'NO'
    })
    print(f'{name:<22} {mean_ms:>8.2f} ms  ±{std_ms:.2f}  RTF={rtf:.4f}')

bench_df = pd.DataFrame(bench_results).sort_values('RTF').reset_index(drop=True)
print('\nBenchmark Results (sorted by RTF):')
display(bench_df)

In [ ]:
# Styled DataFrame and bar chart
fig, ax = plt.subplots(figsize=(10, 5))
colors = ['green' if r < 1.0 else 'red' for r in bench_df['RTF']]
bars = ax.barh(bench_df['Effect'], bench_df['RTF'], color=colors, alpha=0.8)
ax.axvline(1.0, color='black', linestyle='--', linewidth=1.5, label='Real-time boundary (RTF=1)')
for bar, val in zip(bars, bench_df['RTF']):
    ax.text(val + 0.02, bar.get_y() + bar.get_height()/2, f'{val:.3f}',
            va='center', fontsize=9)
ax.set_xlabel('Real-Time Factor (lower = faster)', fontsize=10)
ax.set_title(f'Effect Processing Speed Benchmark ({bench_duration}s audio, {BENCH_ITERS} iterations)', fontsize=11)
ax.legend(); ax.grid(True, alpha=0.3, axis='x')
plt.tight_layout()
plt.savefig(PROCESSED_DIR / 'plot_benchmark_rtf.png', dpi=150, bbox_inches='tight')
plt.show()
print('Benchmark chart saved.')

## Section 4: Latency Analysis — Group Delay

Group delay is the derivative of phase response with respect to frequency: τ(ω) = -dφ/dω. It represents the delay each frequency component experiences through a filter. For an ideal allpass filter, group delay is constant (flat).

In [ ]:
def compute_group_delay(b, a, sr, n_freqs=1024):
    """Compute group delay from IIR filter coefficients."""
    w, gd = scipy_sig.group_delay((b, a), w=n_freqs, fs=sr)
    return w, gd / sr * 1000  # convert to ms

def parametric_eq_filters(sr):
    """Get biquad coefficients for ParametricEQ bands."""
    peq = ParametricEQ(peak1_gain_db=6.0, peak2_gain_db=-4.0,
                       high_shelf_gain_db=3.0, low_shelf_gain_db=2.0)
    bands = [
        ('Low Shelf', *peq._ls(peq.low_shelf_freq, peq.low_shelf_gain_db, sr)),
        ('Peak 1k', *peq._pk(peq.peak1_freq, peq.peak1_gain_db, peq.peak1_q, sr)),
        ('Peak 4k', *peq._pk(peq.peak3_freq, peq.peak3_gain_db, peq.peak3_q, sr)),
        ('High Shelf', *peq._hs(peq.high_shelf_freq, peq.high_shelf_gain_db, sr)),
    ]
    return bands

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ParametricEQ group delay
bands = parametric_eq_filters(SR)
colors = plt.cm.Set1(np.linspace(0, 1, len(bands)))
for (name, b, a), col in zip(bands, colors):
    w, gd = compute_group_delay(b, a, SR)
    axes[0].semilogx(w, gd, label=name, color=col, linewidth=1.5)
axes[0].set_xlim(20, SR/2); axes[0].set_ylim(-5, 5)
axes[0].set_xlabel('Frequency (Hz)', fontsize=10)
axes[0].set_ylabel('Group Delay (ms)', fontsize=10)
axes[0].set_title('ParametricEQ - Group Delay per Band', fontsize=10)
axes[0].legend(fontsize=8); axes[0].grid(True, alpha=0.3, which='both')

# IIR filter group delay for butterworth bandpass (example)
filter_orders = [2, 4, 6, 8]
fc_range = [200/(SR/2), 2000/(SR/2)]
colors2 = plt.cm.Blues(np.linspace(0.4, 0.9, len(filter_orders)))
for order, col in zip(filter_orders, colors2):
    try:
        b, a = scipy_sig.butter(order, fc_range, btype='band')
        w, gd = compute_group_delay(b, a, SR)
        axes[1].semilogx(w, np.clip(gd, 0, 50), label=f'Order {order}', color=col, linewidth=1.5)
    except Exception:
        pass
axes[1].set_xlim(20, SR/2)
axes[1].set_xlabel('Frequency (Hz)', fontsize=10)
axes[1].set_ylabel('Group Delay (ms)', fontsize=10)
axes[1].set_title('Butterworth Bandpass - Group Delay vs Order', fontsize=10)
axes[1].legend(fontsize=8); axes[1].grid(True, alpha=0.3, which='both')
plt.tight_layout()
plt.savefig(PROCESSED_DIR / 'plot_group_delay.png', dpi=150, bbox_inches='tight')
plt.show()
print('Group delay plots saved.')

## Section 5: A/B Comparison Framework

In [ ]:
class EffectsChain:
    def __init__(self): self.effects = []
    def add(self, e): self.effects.append(e); return self
    def process(self, signal, sr):
        x = signal.astype(np.float32)
        for e in self.effects:
            if not getattr(e, 'bypass', False):
                x = e.process(x, sr)
        return x

class ABTest:
    """
    A/B comparison: runs two EffectsChain configs on the same input,
    computes all quality metrics side-by-side, and plots a difference spectrogram.
    """
    def __init__(self, chain_a: EffectsChain, chain_b: EffectsChain,
                 name_a: str = 'Chain A', name_b: str = 'Chain B'):
        self.chain_a = chain_a
        self.chain_b = chain_b
        self.name_a = name_a
        self.name_b = name_b

    def run(self, signal: np.ndarray, sr: int):
        self.dry = signal.astype(np.float32)
        self.wet_a = self.chain_a.process(signal, sr)
        self.wet_b = self.chain_b.process(signal, sr)
        self.sr = sr
        self.metrics_a = compute_all_metrics(self.dry, self.wet_a, sr)
        self.metrics_b = compute_all_metrics(self.dry, self.wet_b, sr)

    def report(self):
        print(f'\nA/B Comparison: {self.name_a} vs {self.name_b}')
        print(f'{"-"*60}')
        print(f'{"Metric":<28} {self.name_a:>14} {self.name_b:>14}')
        print(f'{"-"*60}')
        for key in self.metrics_a:
            a_val = self.metrics_a[key]
            b_val = self.metrics_b[key]
            print(f'{key:<28} {str(a_val):>14} {str(b_val):>14}')

    def plot_diff_spectrogram(self):
        """Plot difference spectrogram: wet_A - wet_B in dB."""
        n = min(len(self.wet_a), len(self.wet_b))
        n_fft = 1024; hop = 256

        def stft_db(x):
            f, t, Z = scipy_sig.stft(x[:n].astype(np.float64), fs=self.sr,
                                       nperseg=n_fft, noverlap=n_fft-hop)
            return f, t, 20*np.log10(np.abs(Z)+1e-10)

        f, t, spec_a = stft_db(self.wet_a)
        _, _, spec_b = stft_db(self.wet_b)
        diff = spec_a - spec_b

        fig, axes = plt.subplots(1, 3, figsize=(15, 4))
        vmin, vmax = -60, 0
        for ax, data, title, cmap in [
            (axes[0], spec_a, self.name_a, 'magma'),
            (axes[1], spec_b, self.name_b, 'magma'),
            (axes[2], diff,   f'{self.name_a} - {self.name_b}', 'RdBu_r')
        ]:
            im = ax.pcolormesh(t, f, data, shading='auto',
                               cmap=cmap, vmin=vmin if cmap=='magma' else -20,
                               vmax=vmax if cmap=='magma' else 20)
            ax.set_yscale('log'); ax.set_ylim(20, self.sr/2)
            ax.set_title(title, fontsize=9)
            ax.set_xlabel('Time (s)', fontsize=8)
            ax.set_ylabel('Freq (Hz)', fontsize=8)
            plt.colorbar(im, ax=ax, label='dB')
        plt.suptitle('A/B Difference Spectrogram', fontsize=11, fontweight='bold')
        plt.tight_layout()
        plt.savefig(PROCESSED_DIR / 'plot_ab_diff_spectrogram.png', dpi=150, bbox_inches='tight')
        plt.show()

# Build two chains
chain_a = EffectsChain()
chain_a.add(SchroederReverb(room_size=0.3, wet_mix=0.3))
chain_a.add(Compressor(threshold_db=-20, ratio=3))

chain_b = EffectsChain()
chain_b.add(SchroederReverb(room_size=0.8, wet_mix=0.6))
chain_b.add(Distortion(drive=3, post_gain=0.7))

t = np.linspace(0, 2.0, int(SR*2), endpoint=False)
ref = (0.7*np.sin(2*np.pi*440*t) + 0.1*np.random.randn(len(t))).astype(np.float32)

ab = ABTest(chain_a, chain_b, 'Small Room + Comp', 'Large Hall + Dist')
ab.run(ref, SR)
ab.report()
ab.plot_diff_spectrogram()

## Section 6: Batch Evaluation

We run all effects on available audio files, compute metrics, and export results to a heatmap and CSV.

In [ ]:
def load_audio(path):
    data, sr = sf.read(str(path), dtype='float32')
    if data.ndim > 1: data = data[:, 0]
    return data, sr

def get_test_files(max_files=10):
    wavs = list(RAW_DIR.glob('dry_*.wav'))[:max_files]
    if not wavs:
        # Generate fallback signals
        RAW_DIR.mkdir(parents=True, exist_ok=True)
        t = np.linspace(0, 2.0, int(SR*2), endpoint=False)
        signals = {
            'dry_sine_440': (0.7*np.sin(2*np.pi*440*t)).astype(np.float32),
            'dry_pink': np.random.randn(int(SR*2)).astype(np.float32) * 0.3,
        }
        for name, s in signals.items():
            sf.write(str(RAW_DIR/f'{name}.wav'), s, SR, subtype='PCM_16')
        wavs = list(RAW_DIR.glob('dry_*.wav'))[:max_files]
    return wavs

test_files = get_test_files(5)  # Limit for speed
print(f'Testing on {len(test_files)} files x {len(ALL_EFFECTS)} effects...')

batch_rows = []
for wav_path in test_files:
    dry, sr = load_audio(wav_path)
    sig_name = wav_path.stem
    for effect in ALL_EFFECTS:
        eff_name = effect.__class__.__name__
        wet = effect.process(dry, sr)
        m = compute_all_metrics(dry, wet, sr)
        row = {'file': sig_name, 'effect': eff_name}
        row.update(m)
        batch_rows.append(row)

batch_df = pd.DataFrame(batch_rows)
eval_csv = BASE_DIR / 'data' / 'evaluation_results.csv'
batch_df.to_csv(str(eval_csv), index=False)
print(f'Evaluation results saved: {eval_csv}')
print(f'Total rows: {len(batch_df)}')
display(batch_df.head(8))

In [ ]:
# Mean metrics per effect
metric_cols = ['snr_db', 'thd_pct', 'pesq_approx', 'centroid_shift_hz', 'crest_wet', 'dynrange_wet_db']
agg = batch_df.groupby('effect')[metric_cols].mean()

# Normalize each metric to [0, 1] for heatmap
agg_norm = agg.copy()
for col in metric_cols:
    col_range = agg[col].max() - agg[col].min()
    if col_range > 0:
        agg_norm[col] = (agg[col] - agg[col].min()) / col_range
    else:
        agg_norm[col] = 0.5

fig, ax = plt.subplots(figsize=(12, 6))
im = ax.imshow(agg_norm.values, cmap='RdYlGn', aspect='auto', vmin=0, vmax=1)
ax.set_xticks(range(len(metric_cols)))
ax.set_yticks(range(len(agg_norm)))
ax.set_xticklabels(metric_cols, rotation=30, ha='right', fontsize=9)
ax.set_yticklabels(agg_norm.index, fontsize=9)
plt.colorbar(im, ax=ax, label='Normalized Score (0=low, 1=high)')
# Annotate cells
for i in range(len(agg_norm)):
    for j in range(len(metric_cols)):
        raw_val = agg.values[i, j]
        ax.text(j, i, f'{raw_val:.2f}', ha='center', va='center', fontsize=7)
ax.set_title('Effects x Metrics Heatmap (mean over all test signals)', fontsize=11, fontweight='bold')
plt.tight_layout()
plt.savefig(PROCESSED_DIR / 'plot_eval_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()
print('Heatmap saved.')

In [ ]:
print('Mean ± Std by Effect:')
for col in ['snr_db', 'thd_pct', 'pesq_approx']:
    mean_std = batch_df.groupby('effect')[col].agg(['mean','std']).round(3)
    print(f'\n{col}:')
    print(mean_std.to_string())

In [ ]:
print('=' * 60)
print('NOTEBOOK 04 COMPLETE')
print('=' * 60)
print(f'Quality metrics: SNR, THD, PESQ-approx, spectral centroid, crest, dynrange')
print(f'Benchmark: {len(ALL_EFFECTS)} effects x {BENCH_ITERS} iterations on {bench_duration}s audio')
print(f'Group delay: ParametricEQ bands + Butterworth orders')
print(f'A/B framework: difference spectrogram + side-by-side metrics')
print(f'Batch eval: {len(test_files)} files x {len(ALL_EFFECTS)} effects = {len(batch_df)} rows')
print(f'Exports: evaluation_results.csv, heatmap PNG, benchmark PNG')
print('\nCheckmark: Notebook complete')